# Modulo 2 - Indice temporal, frecuencia y train/test split en Python

Curso: **Series Temporales con R y Python** (Frogames Formacion).

Codigo de acompanamiento de la leccion `02-manejo-de-series-en-r-y-python/01-indice-temporal-frecuencia-y-split` -- abrelo mientras lees la leccion y ejecuta cada celda tu mismo.

Trabajamos con el cierre diario de **cuatro indices bursatiles** (S&P 500, DAX, FTSE 100, Nikkei 225), descargados con `yfinance` del 2015-01-02 al 2026-09-15. El CSV ya esta en esta carpeta (`indices_2015_2026.csv`).

In [1]:
# Opcional: asi se genero indices_2015_2026.csv. No hace falta ejecutar esta celda
# para seguir el resto del notebook (el CSV ya esta en esta carpeta).

import yfinance as yf

raw = yf.download(tickers="^GSPC ^FTSE ^N225 ^GDAXI", start="2015-01-01", end="2026-09-16",
                   interval="1d", group_by="ticker", auto_adjust=True, threads=False)
# df_comp = raw.copy()
# df_comp['spx'] = df_comp['^GSPC'].Close
# df_comp['dax'] = df_comp['^GDAXI'].Close
# df_comp['ftse'] = df_comp['^FTSE'].Close
# df_comp['nikkei'] = df_comp['^N225'].Close
# df_comp = df_comp[['spx','dax','ftse','nikkei']]
# df_comp.to_csv('indices_2015_2026.csv', index_label='date')

[                       0%                       ]

[**********************50%                       ]  2 of 4 completed

[**********************75%***********            ]  3 of 4 completed

[*********************100%***********************]  4 of 4 completed

[*********************100%***********************]  4 of 4 completed

## Carga y primer vistazo

In [2]:
import pandas as pd
import numpy as np

raw_csv_data = pd.read_csv("indices_2015_2026.csv")
df_comp = raw_csv_data.copy()
df_comp.head()

,date,spx,dax,ftse,nikkei
0,2015-01-02,2058.199951,9764.730469,6547.799805,NaN
1,2015-01-05,2020.579956,9473.160156,6417.200195,17408.710938
2,2015-01-06,2002.609985,9469.660156,6366.500000,16883.189453
3,2015-01-07,2025.900024,9518.179688,6419.799805,16885.330078
4,2015-01-08,2062.139893,9837.610352,6570.000000,17167.099609


In [3]:
df_comp.dtypes

date          str
spx       float64
dax       float64
ftse      float64
nikkei    float64
dtype: object

## De texto a fecha, y fecha a indice

In [4]:
df_comp.date = pd.to_datetime(df_comp.date)
df_comp.set_index("date", inplace=True)
df_comp.head()

,spx,dax,ftse,nikkei
date,,,,
2015-01-02,2058.199951,9764.730469,6547.799805,NaN
2015-01-05,2020.579956,9473.160156,6417.200195,17408.710938
2015-01-06,2002.609985,9469.660156,6366.500000,16883.189453
2015-01-07,2025.900024,9518.179688,6419.799805,16885.330078
2015-01-08,2062.139893,9837.610352,6570.000000,17167.099609


## Fijando la frecuencia: D vs B

`asfreq('D')` inserta una fila por cada dia natural (incluidos fines de semana); `asfreq('B')` solo inserta dias laborables. Para un indice bursatil, `B` es la eleccion correcta -- `D` mete NaN todos los sabados y domingos, que nunca van a tener dato.

In [5]:
df_d = df_comp.asfreq('D')
print(df_d.shape)
df_d.head(8)

(4275, 4)


,spx,dax,ftse,nikkei
date,,,,
2015-01-02,2058.199951,9764.730469,6547.799805,NaN
2015-01-03,NaN,NaN,NaN,NaN
2015-01-04,NaN,NaN,NaN,NaN
2015-01-05,2020.579956,9473.160156,6417.200195,17408.710938
2015-01-06,2002.609985,9469.660156,6366.500000,16883.189453
2015-01-07,2025.900024,9518.179688,6419.799805,16885.330078
2015-01-08,2062.139893,9837.610352,6570.000000,17167.099609
2015-01-09,2044.810059,9648.500000,6501.100098,17197.730469


In [6]:
df_comp = df_comp.asfreq('B')
print(df_comp.shape)
df_comp.head(8)

(3053, 4)


,spx,dax,ftse,nikkei
date,,,,
2015-01-02,2058.199951,9764.730469,6547.799805,NaN
2015-01-05,2020.579956,9473.160156,6417.200195,17408.710938
2015-01-06,2002.609985,9469.660156,6366.500000,16883.189453
2015-01-07,2025.900024,9518.179688,6419.799805,16885.330078
2015-01-08,2062.139893,9837.610352,6570.000000,17167.099609
2015-01-09,2044.810059,9648.500000,6501.100098,17197.730469
2015-01-12,2028.260010,9781.900391,6501.399902,NaN
2015-01-13,2023.030029,9941.000000,6542.200195,17087.710938


## Valores faltantes

Con `asfreq('B')` aparecen NaN reales: festivos bursatiles de un mercado que no coinciden con los de otro (Wall Street cierra en dias que Tokio no, y viceversa).

In [7]:
df_comp.isna().sum()

spx       111
dax        83
ftse       97
nikkei    193
dtype: int64

In [8]:
df_comp.spx = df_comp.spx.ffill()   # ultimo valor conocido hacia adelante
df_comp.ftse = df_comp.ftse.bfill()  # primer valor conocido hacia atras
df_comp.dax = df_comp.dax.fillna(value=df_comp.dax.mean())  # con la media (solo con fines didacticos)
df_comp.nikkei = df_comp.nikkei.ffill().bfill()
df_comp.isna().sum()

spx       0
dax       0
ftse      0
nikkei    0
dtype: int64

## Simplificando el dataset

A partir de aqui, el resto del curso trabaja sobre una unica serie: nos quedamos con el S&P 500 y lo renombramos `market_value`.

In [9]:
df_comp['market_value'] = df_comp.spx
del df_comp['spx']
del df_comp['dax']
del df_comp['ftse']
del df_comp['nikkei']
df_comp.describe()

,market_value
count,3053.000000
mean,3833.557190
std,1548.500395
min,1829.079956
25%,2569.129883
50%,3545.530029
75%,4649.229980
max,7798.990234


## Train / test split respetando el orden temporal

Nunca `train_test_split` aleatorio: el 80% mas antiguo es train, el 20% mas reciente es test.

In [10]:
size = int(len(df_comp) * 0.8)
df = df_comp.iloc[:size]
df_test = df_comp.iloc[size:]
print("total:", len(df_comp), "train:", len(df), "test:", len(df_test))
df.tail()

total: 3053 train: 2442 test: 611


,market_value
date,
2024-05-07,5187.700195
2024-05-08,5187.669922
2024-05-09,5214.080078
2024-05-10,5222.680176
2024-05-13,5221.419922


In [11]:
df_test.head()

,market_value
date,
2024-05-14,5246.680176
2024-05-15,5308.149902
2024-05-16,5297.100098
2024-05-17,5303.270020
2024-05-20,5308.129883


**Entregable**: reproduce estas celdas con el CSV incluido. Cambia el `0.8` del split por `0.9` y `0.7` y observa como cambia el tamano de `df_test` -- es la base del walk-forward validation que se vera en el Modulo 8 (ARIMA).